# Quickstart

A simple guide to computing EVI zonal statistics using Google Earth Engine or local computation.

In [ ]:
import evy
import attaviz

attaviz.enable()

## Load Administrative Boundaries

`evy` aggregates EVI over polygons you supply. Use `get_boundaries` to fetch administrative units from [GeoBoundaries](https://www.geoboundaries.org/) (cached locally after the first call), or `load_boundaries` to read your own shapefile, GeoJSON, or GeoPackage.

In [ ]:
gdf = evy.get_boundaries("SMR", admin_level=1)
gdf.head()

## Basic Usage

Pass the `GeoDataFrame` to `zonal_stats` along with `zone_col`, which is the column in your boundaries that identifies each zone. For GeoBoundaries data that's `shapeName`. The simplest call returns monthly EVI means for the past year:

In [ ]:
df = evy.zonal_stats(gdf, zone_col="shapeName")
df.head()

## Choose Data Source

The default data source is MODIS at 250m resolution. By default, zonal statistics are masked to cropland areas using Dynamic World land cover. Disable this with `mask_cropland=False`. 

Currently, we only support a single land cover dataset, but we're planning to add more in the future.

In [ ]:
df_modis = evy.zonal_stats(
    gdf,
    zone_col="shapeName",
    source="modis",
    start_date="2024-01-01",
    end_date="2024-12-31",
    freq=evy.MONTHLY,  # or freq="ME"
    stats=["mean", "std", "min", "max"],
)

df_modis.head()

We also support Sentinel-2 for higher (10m) resolution:

In [ ]:
df_s2 = evy.zonal_stats(
    gdf,
    zone_col="shapeName",
    source="sentinel2",
    start_date="2024-01-01",
    end_date="2024-12-31",
    freq=evy.MONTHLY,
)

df_s2.head()

## Local Backend (No GEE Required)

Set `backend="local"` to compute EVI zonal statistics without Google Earth Engine. This uses Microsoft's Planetary Computer STAC catalog and it requires no authentication. Currently, the local backend only supports MODIS EVI with ESA WorldCover cropland masking, but we're planning to add Sentinel-2 and more land cover datasets in the future.

The local backend method consists of searching for MODIS data via STAC, loading rasters lazily with `odc-stac`, applying quality masking and EVI scaling, masking cropland using ESA WorldCover (class 40), and computing zonal stats with `exactextract` (partial-pixel weighting).

Both backends return the same columns: `date` (start of each period), your `zone_col`, and one column per statistic (`mean`, `std`, ...).

In [ ]:
df_local = evy.zonal_stats(
    gdf,
    zone_col="shapeName",
    backend="local",
    start_date="2023-01-01",
    end_date="2023-12-31",
    freq=evy.MONTHLY,
    stats=["mean", "median"],
    include_geometry=True,
)

df_local.head()

## Low-Level Local Pipeline

For full control over the local computation pipeline, e.g. to inspect the raw EVI raster or the raw cropland mask, you can call the three main steps separately:

1. `load_modis`: STAC search + lazy `xarray.Dataset` with `evi_raw` and `qa` bands
2. `load_landcover`: aligned ESA WorldCover raster for cropland masking
3. `compute_zonal_stats`: full processing pipeline (quality mask → scale → temporal aggregation → optional cropland mask → extraction)

This is equivalent to `zonal_stats(backend="local")` but exposes the intermediate `xarray` objects.

In [ ]:
ds_evi = evy.load_modis(gdf, start_date="2023-01-01", end_date="2023-12-31")
land_cover = evy.load_landcover(gdf, ds_evi)

df_pipeline = evy.compute_zonal_stats(
    ds_evi,
    gdf,
    zone_col="shapeName",
    land_cover=land_cover,
    freq=evy.MONTHLY,
    stats=["mean", "std"],
)
df_pipeline.head()

## Custom Boundaries

Use your own shapefile, GeoJSON, or GeoPackage with `evy.load_boundaries`. Pass the name of any column that identifies each zone as `zone_col`:

In [ ]:
# gdf_custom = evy.load_boundaries("path/to/your/boundaries.shp")
# df_custom = evy.zonal_stats(
#     gdf_custom,
#     zone_col="admin_name",  # any column in your file
#     source="modis",
#     start_date="2024-01-01",
#     end_date="2024-12-31",
#     freq=evy.MONTHLY,
# )

## Large Jobs - Export to Google Drive

For queries spanning many years, export to Drive instead of downloading directly. Only supported with `backend="gee"`.

In [ ]:
# This starts an async export task
# task_id = evy.zonal_stats(
#     gdf,
#     zone_col="shapeName",
#     start_date="2010-01-01",
#     end_date="2024-12-31",
#     freq=evy.YEARLY,
#     mask_cropland=True,
#     export_to_drive=True,
#     drive_folder="evy_exports",
# )
#
# print(f"Export started! Task ID: {task_id}")
# print("Check Google Drive for results when complete.")

# Check task status
# status = evy.check_task_status(task_id)
# print(status)

## Available Collections

See what data sources are available:

In [ ]:
evy.list_collections()

## Phenology Extraction (SOS, MOS, EOS)

We provide a phenology extraction method to extract crop seasonality metrics: Start of Season (SOS), Middle of Season (MOS), and End of Season (EOS).

The `value_col` argument must match the stat column in your input, for example `"mean"`. The name is the same for both backends.

In [ ]:
phenology = evy.calculate_phenology(df_modis, value_col="mean")
phenology.head()

Calculate the phenology metrics per governorate:

In [ ]:
phenology_by_region = evy.calculate_phenology(
    df_modis, value_col="mean", group_col="shapeName"
)
phenology_by_region.head(12)

Filter data to the growing season only:

In [ ]:
df_growing = evy.filter_growing_season(df_modis, start_month=2, end_month=6)
df_growing.head()

## Visualizations

Built-in interactive charts using Altair:

In [ ]:
chart = evy.plot_seasonality(phenology)
chart

In [ ]:
chart_regions = evy.plot_seasonality_by_region(
    phenology_by_region, region_col="shapeName"
)
chart_regions